# 03 - Feature engineering
`ChurnFeatureEngineer` builds 12 churn-relevant features from each customer's own row (stateless, so it is leakage-safe). See `src/features/feature_engineering.py` for the definition and reasoning of every feature.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
import pandas as pd
import numpy as np
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
from src import config
config.ensure_directories()

In [2]:
from src.data.load_data import load_raw_data
from src.data.clean_data import clean_raw_dataframe, split_features_target
from src.features.feature_engineering import ChurnFeatureEngineer, ENGINEERED_FEATURES
df, _ = clean_raw_dataframe(load_raw_data())
X, y, ids = split_features_target(df)
fe = ChurnFeatureEngineer().fit(X)
Xe = fe.transform(X)
print(len(ENGINEERED_FEATURES), 'engineered features:', ENGINEERED_FEATURES)
Xe[['tenure', 'MonthlyCharges', 'TotalCharges'] + ENGINEERED_FEATURES].head(8)

12 engineered features: ['service_count', 'protection_service_count', 'streaming_service_count', 'avg_monthly_charge', 'charge_delta', 'charges_per_service', 'is_month_to_month', 'auto_payment', 'is_electronic_check', 'has_family', 'mtm_fiber', 'tenure_group']


,tenure,MonthlyCharges,TotalCharges,service_count,protection_service_count,streaming_service_count,avg_monthly_charge,charge_delta,charges_per_service,is_month_to_month,auto_payment,is_electronic_check,has_family,mtm_fiber,tenure_group
0,1,29.85,29.85,2,1,0,29.850000,0.000000,14.925000,1,0,1,1,0,0-12
1,34,56.95,1889.50,4,2,0,55.573529,1.376471,14.237500,0,0,0,0,0,25-48
2,2,53.85,108.15,4,2,0,54.075000,-0.225000,13.462500,1,0,0,0,0,0-12
3,45,42.30,1840.75,4,3,0,40.905556,1.394444,10.575000,0,1,0,0,0,25-48
4,2,70.70,151.65,2,0,0,75.825000,-5.125000,35.350000,1,0,1,0,1,0-12
5,8,99.65,820.50,6,1,2,102.562500,-2.912500,16.608333,1,0,1,0,1,0-12
6,22,89.10,1949.40,5,1,1,88.609091,0.490909,17.820000,1,1,0,1,1,13-24
7,10,29.75,301.90,2,1,0,30.190000,-0.440000,14.875000,1,0,0,0,0,0-12


### Sanity checks
1. The transformer is stateless: transforming a single row gives the same values as transforming the whole table.
2. Engineered features relate to churn in the expected direction (association only).

In [3]:
row = X.iloc[[10]]
assert fe.transform(row)[ENGINEERED_FEATURES].equals(Xe.iloc[[10]][ENGINEERED_FEATURES]) or np.allclose(
    fe.transform(row)[[c for c in ENGINEERED_FEATURES if c != 'tenure_group']].astype(float).values,
    Xe.iloc[[10]][[c for c in ENGINEERED_FEATURES if c != 'tenure_group']].astype(float).values, equal_nan=True)
print('Row-wise transform matches full-table transform: OK')

Row-wise transform matches full-table transform: OK


In [4]:
Xe['Churn'] = y.values
num = [c for c in ENGINEERED_FEATURES if c != 'tenure_group']
Xe[num + ['Churn']].corr()['Churn'].drop('Churn').sort_values().round(3)

auto_payment               -0.210
protection_service_count   -0.173
has_family                 -0.163
service_count              -0.019
charge_delta                0.002
streaming_service_count     0.071
avg_monthly_charge          0.193
charges_per_service         0.292
is_electronic_check         0.302
is_month_to_month           0.405
mtm_fiber                   0.418
Name: Churn, dtype: float64

In [5]:
from src.data.preprocess import build_preprocessor
from src.models.model_training import make_pipeline
from sklearn.linear_model import LogisticRegression
pipe = make_pipeline(LogisticRegression(max_iter=500)).fit(X, y)
names = pipe[:-1].get_feature_names_out()
print(len(names), 'model input columns after one-hot encoding, e.g.:', list(names[:8]))

61 model input columns after one-hot encoding, e.g.: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'service_count', 'protection_service_count', 'streaming_service_count', 'avg_monthly_charge']
